# Vectorless RAG System - Complete Tutorial

This notebook demonstrates the complete vectorless RAG system with supervised learning components.

## Contents
1. Setup and Installation
2. Document Indexing
3. Basic Querying
4. Supervised Dataset Creation
5. System Evaluation
6. Domain Rule Learning

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install openai pypdf2 python-dotenv numpy

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Add src to path
sys.path.append('../src')

# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError("Please set OPENAI_API_KEY in .env file")

print("✅ Environment configured")

## 2. Document Indexing

### Concept Explanation

**What is happening**: The system reads your PDF page by page and builds a hierarchical tree structure.

**Why this matters**: Instead of chopping the document into arbitrary 512-token chunks (which destroys structure), we respect the document's natural organization.

**How it works**:
1. **Extract pages**: Read PDF page-by-page using PyPDF2
2. **Detect structure**: LLM analyzes first 20 pages for Table of Contents
3. **Build tree**: Create hierarchical index with chapters → sections → subsections
4. **Generate summaries**: LLM creates brief summaries for each node

**Output**: A tree where each node knows:
- Its title (e.g., "Methodology")
- Page range (e.g., pages 10-15)
- Its children (e.g., "Data Collection", "Analysis")
- A summary of its content

In [ ]:
from vectorless_rag import VectorlessRAG

# Initialize RAG system
rag = VectorlessRAG(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-4o-mini"  # Fast and cost-effective
)

print("✅ RAG system initialized")

In [ ]:
# Index your document
# Replace with your PDF path
PDF_PATH = "../data/sample_document.pdf"
CACHE_PATH = "../data/sample_document_tree.json"

print("📄 Indexing document...")
print("This may take 1-2 minutes depending on document length\n")

tree = rag.index_document(
    pdf_path=PDF_PATH,
    cache_path=CACHE_PATH,
    max_pages_per_node=10,
    generate_summaries=True
)

print("\n✅ Document indexed!")

In [ ]:
# Visualize the tree structure
print("\n🌲 Document Tree Structure:\n")
print("="*80)
rag.print_tree()
print("="*80)

## 3. Basic Querying

### Concept Explanation

**What is happening**: The LLM looks at your question and the tree structure, then reasons about which sections are relevant.

**Why this is different from vector search**:
- **Vector RAG**: Embeds question → finds chunks with similar embeddings → hopes they're relevant
- **Tree RAG**: LLM reads question → understands intent → logically determines which sections to retrieve

**Example**:
- Question: "What were the main findings?"
- Vector RAG might return: Any chunk mentioning "findings" (could be "future work" section)
- Tree RAG reasons: "Main findings = Results or Discussion section" → retrieves those specific nodes

**The retrieval process**:
1. Convert tree to text representation
2. LLM reads: question + tree structure
3. LLM reasons: "This question needs nodes X, Y, Z because..."
4. Return node IDs with reasoning

**Output**: Specific sections with clear rationale

In [ ]:
# Simple query
question = "What is the main objective of this document?"

print(f"\n❓ Question: {question}\n")
print("="*80)

result = rag.query(
    question=question,
    max_nodes=3
)

print("\n📝 Answer:")
print(result['answer'])

print("\n📚 Sources:")
for source in result['sources']:
    print(f"  • {source['title']} (pages {source['pages']})")

print("\n" + "="*80)

In [ ]:
# Query with domain rules
# This demonstrates expert-guided retrieval

financial_rules = """
DOMAIN RULES FOR FINANCIAL DOCUMENTS:
1. For revenue/income questions:
   - Check Income Statement first
   - Then MD&A (Management Discussion & Analysis)
2. For risk-related questions:
   - Prioritize Risk Factors section
   - Also check MD&A
3. For accounting policy questions:
   - Look in Notes to Financial Statements
"""

question = "What are the main revenue sources?"

print(f"\n❓ Question (with domain rules): {question}\n")
print("="*80)

result = rag.query(
    question=question,
    max_nodes=5,
    domain_rules=financial_rules  # Inject expert knowledge
)

print("\n📝 Answer:")
print(result['answer'])

print("\n" + "="*80)

## 4. Supervised Dataset Creation

### Concept Explanation

**What is supervised learning in RAG?**

Create a "test set" of question-answer pairs where you know the correct answer and which sections should be retrieved.

**Why this matters**:
- Measure system accuracy objectively
- Identify where retrieval fails
- Learn patterns to improve the system

**Each QA pair contains**:
1. **Question**: The query
2. **Ground truth answer**: What the correct answer should be
3. **Relevant nodes**: Which sections should be retrieved
4. **Context needed**: What information is required
5. **Difficulty**: Easy/medium/hard

**Example**:
```json
{
  "question": "What was the sample size?",
  "ground_truth_answer": "500 participants",
  "relevant_node_ids": ["node_0005"],
  "context_needed": "Methodology section",
  "difficulty": "easy"
}
```

In [ ]:
from supervised_rag import SupervisedRAGEvaluator

# Initialize evaluator
evaluator = SupervisedRAGEvaluator(
    openai_client=rag.client,
    model="gpt-4o-mini"
)

print("✅ Evaluator initialized")

In [ ]:
# Add supervised QA pairs
# Customize these for your document!

# Example 1: Easy question
evaluator.add_qa_pair(
    question="What is the document about?",
    ground_truth_answer="This document presents research on [topic]",
    relevant_node_ids=["node_0002"],  # Introduction node
    context_needed="Introduction section",
    difficulty="easy",
    category="general"
)

# Example 2: Medium question
evaluator.add_qa_pair(
    question="What methodology was employed?",
    ground_truth_answer="The study used [method details]",
    relevant_node_ids=["node_0004", "node_0005"],  # Methodology nodes
    context_needed="Methodology section",
    difficulty="medium",
    category="methods"
)

# Example 3: Hard question (multi-hop)
evaluator.add_qa_pair(
    question="How do the results compare to previous studies?",
    ground_truth_answer="[Comparison details]",
    relevant_node_ids=["node_0007", "node_0003"],  # Results + Lit Review
    context_needed="Results and Literature Review sections",
    difficulty="hard",
    category="analysis"
)

print(f"✅ Added {len(evaluator.qa_pairs)} QA pairs")

In [ ]:
# Save QA dataset for reuse
QA_DATASET_PATH = "../supervised/qa_dataset.json"

evaluator.save_qa_pairs(QA_DATASET_PATH)

print(f"💾 Saved dataset to {QA_DATASET_PATH}")

## 5. System Evaluation

### Concept Explanation

**What are we measuring?**

Two things:
1. **Retrieval Quality**: Did we get the right sections?
2. **Answer Quality**: Is the answer correct?

**Retrieval Metrics**:

1. **Precision**: Of the sections we retrieved, how many were actually relevant?
   - Formula: `relevant_retrieved / total_retrieved`
   - Example: Retrieved 3 nodes, 2 were relevant → Precision = 2/3 = 0.67

2. **Recall**: Of all relevant sections, how many did we retrieve?
   - Formula: `relevant_retrieved / total_relevant`
   - Example: 2 relevant nodes exist, we got 1 → Recall = 1/2 = 0.50

3. **F1 Score**: Harmonic mean of precision and recall
   - Formula: `2 * (P * R) / (P + R)`
   - Balances precision and recall

4. **MRR (Mean Reciprocal Rank)**: How early did we get a relevant result?
   - First result is relevant → MRR = 1.0
   - Second result is relevant → MRR = 0.5

**Answer Metrics**:

1. **ROUGE-1**: Word overlap between generated and ground truth
2. **ROUGE-L**: Longest common subsequence
3. **Semantic Similarity**: Cosine similarity of embeddings (0-1)

In [ ]:
# Run full evaluation
print("\n" + "="*80)
print("🔬 RUNNING EVALUATION")
print("="*80 + "\n")

results = evaluator.evaluate_rag_system(
    rag_system=rag,
    verbose=True  # Show individual results
)

print("\n" + "="*80)
print("✅ EVALUATION COMPLETE")
print("="*80)

In [ ]:
# View aggregate metrics
print("\n" + "="*80)
print("📊 AGGREGATE PERFORMANCE")
print("="*80)
print(results['aggregate_metrics']['summary'])
print("="*80)

In [ ]:
# Analyze per-question performance
import pandas as pd

# Convert results to DataFrame for easy analysis
rows = []
for r in results['per_question_results']:
    if 'error' in r:
        continue
    rows.append({
        'Question': r['question'][:50] + '...',
        'Difficulty': r['difficulty'],
        'Precision': r['retrieval_metrics']['precision'],
        'Recall': r['retrieval_metrics']['recall'],
        'F1': r['retrieval_metrics']['f1_score'],
        'ROUGE-1': r['answer_metrics']['rouge_1'],
        'Semantic Sim': r['answer_metrics']['semantic_similarity']
    })

df = pd.DataFrame(rows)
print("\n📋 Per-Question Results:\n")
print(df.to_string(index=False))

## 6. Domain Rule Learning

### Concept Explanation

**What is this doing?**

When the system makes mistakes (low F1 scores), we want to understand WHY and learn from it.

**The process**:
1. Identify failed queries (F1 < 0.5)
2. Send failure patterns to LLM
3. LLM analyzes and suggests domain-specific rules

**Example**:

**Failure**: Question about "depreciation method" retrieved Income Statement instead of Notes to Financial Statements

**LLM Analysis**: "Accounting policy questions require checking Notes section"

**Suggested Rule**: "For accounting policy questions (depreciation, revenue recognition), always check Notes to Financial Statements"

**How to use the rule**: Add it to `domain_rules` parameter in your next queries!

In [ ]:
from supervised_rag import DomainRuleLearner

# Initialize rule learner
learner = DomainRuleLearner(
    openai_client=rag.client,
    model="gpt-4o-mini"
)

# Analyze failures
print("\n" + "="*80)
print("🔍 ANALYZING FAILURE PATTERNS")
print("="*80 + "\n")

analysis = learner.analyze_failures(results)

print(analysis)
print("\n" + "="*80)

## 7. Using Learned Rules

After analyzing failures, incorporate suggested rules into your queries:

In [ ]:
# Example: Apply learned domain rules

improved_rules = """
LEARNED DOMAIN RULES:
1. [Add rules suggested by failure analysis here]
2. [Example: For methodology questions, check Methods section first]
3. [Example: For comparison questions, check both Results and Discussion]
"""

# Re-run problematic queries with rules
question = "[Insert previously failed question]"

result = rag.query(
    question=question,
    max_nodes=5,
    domain_rules=improved_rules
)

print(f"\n📝 Improved Answer:\n{result['answer']}")

## Summary

You've now learned:

1. ✅ **Document Indexing**: Build hierarchical tree from PDF
2. ✅ **LLM-Based Retrieval**: Reasoning over structure instead of similarity
3. ✅ **Supervised Evaluation**: Measure performance objectively
4. ✅ **Failure Analysis**: Learn from mistakes
5. ✅ **Domain Rules**: Inject expert knowledge without retraining

### Next Steps for Your Projects:

1. **Replace the PDF**: Use your own documents
2. **Create QA pairs**: Build supervised dataset for your domain
3. **Iterate on rules**: Analyze failures → add rules → re-evaluate
4. **Customize**: Adjust parameters (max_pages_per_node, max_nodes, etc.)
5. **Deploy**: Integrate into your application

### Key Advantages Over Vector RAG:

- 📍 **Traceable**: Every answer cites specific sections and pages
- 🎯 **Accurate**: Reasoning beats similarity (98.7% vs 80% on FinanceBench)
- 🧠 **Explainable**: LLM shows its reasoning
- 🔧 **Customizable**: Add domain rules without retraining
- 🏗️ **Structured**: Respects document organization